# XGBoost tuning


This notebook is used to find the best hyperparameters for XGBoost from the train data only.

Evaluation criteria:
- Metric: mean MAE (in MW) across 4 walk-forward folds (`TimeSeriesSplit`)
- Each fold trains on earlier days and validates on the next block, so no future data is used
- The combination with the lowest mean CV MAE is selected
- The fold-to-fold std is reported to show whether differences are real or just noise
- Calibration and test data are not used here

Best hyperparameter is chosen based on loowest CV MAE.

In [1]:
# 1. Load TRAIN only (calib and test are never touched while tuning)
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import itertools
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from config import FEATURES_DATA_DIR, MODELS_DIR

train = pd.read_parquet(FEATURES_DATA_DIR / "features_train.parquet").dropna().reset_index(drop=True)
cols = [c for c in train.columns if c not in ("ds", "y")]
Xtr, ytr = train[cols], train["y"]
print(Xtr.shape, train["ds"].min(), train["ds"].max())

(824, 20) 2021-09-29 00:00:00 2023-12-31 00:00:00


In [2]:
# 2. Search space (shallow trees: ~1,000 daily rows overfit easily)
grid = {
    "max_depth": [1, 2, 3],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [300, 600, 1000, 1500],
    "min_child_weight": [3, 5, 10],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

In [3]:
# 3. Walk-forward CV: each fold trains on the past and validates on the next block (no shuffling)
tscv = TimeSeriesSplit(n_splits=4)

results = []
for vals in itertools.product(*grid.values()):
    params = dict(zip(grid.keys(), vals))
    maes = []
    for tr_i, va_i in tscv.split(Xtr):
        m = XGBRegressor(**params, random_state=42, n_jobs=-1)
        m.fit(Xtr.iloc[tr_i], ytr.iloc[tr_i])
        maes.append(np.mean(np.abs(ytr.iloc[va_i] - m.predict(Xtr.iloc[va_i]))))
    results.append({"params": params, "cv_mae": np.mean(maes), "cv_std": np.std(maes)})

results = sorted(results, key=lambda r: r["cv_mae"])
pd.DataFrame(results).head(10)

,params,cv_mae,cv_std
0,"{'max_depth': 1, 'learning_rate': 0.05, 'n_est...",4715.101883,542.038446
1,"{'max_depth': 1, 'learning_rate': 0.05, 'n_est...",4746.807900,528.195013
2,"{'max_depth': 1, 'learning_rate': 0.05, 'n_est...",4778.650676,601.359849
3,"{'max_depth': 1, 'learning_rate': 0.05, 'n_est...",4782.101772,637.876650
4,"{'max_depth': 1, 'learning_rate': 0.05, 'n_est...",4792.437503,506.819236
5,"{'max_depth': 1, 'learning_rate': 0.1, 'n_esti...",4792.870639,629.482506
6,"{'max_depth': 1, 'learning_rate': 0.05, 'n_est...",4801.781952,661.970461
7,"{'max_depth': 1, 'learning_rate': 0.1, 'n_esti...",4816.026267,566.284316
8,"{'max_depth': 1, 'learning_rate': 0.05, 'n_est...",4818.520938,577.719802
9,"{'max_depth': 1, 'learning_rate': 0.05, 'n_est...",4829.967365,554.690371


In [4]:
# 4. Best params. If one sits on the edge of the grid, widen the grid in that direction and rerun.
best = results[0]["params"]
print(best, "| CV MAE:", round(results[0]["cv_mae"], 1), "+/-", round(results[0]["cv_std"], 1))

{'max_depth': 1, 'learning_rate': 0.05, 'n_estimators': 1000, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8} | CV MAE: 4715.1 +/- 542.0


In [6]:
# 5. Save for the training script in src/
print(json.dumps(best, indent=2))

{
  "max_depth": 1,
  "learning_rate": 0.05,
  "n_estimators": 1000,
  "min_child_weight": 5,
  "subsample": 0.8,
  "colsample_bytree": 0.8
}
